# ST554 Final Project: Siona Benjamin
For the final project we'll use spark to handle streaming data and fitting a machine learning model. The data used below describes power consumption from different time zones of Tetoauan city in relation to factors such as time of day, temperature, and humidity. 

To get started, we'll read in our data as a pandas dataframe, and then convert this to a spark dataframe.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, CrossValidatorModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression
from pyspark.sql.types import StructType
from pyspark.sql.functions import col
from pyspark.ml.feature import SQLTransformer, PCA, Binarizer, OneHotEncoder, VectorAssembler, StringIndexer

In [2]:
#create spark session
spark = SparkSession.builder.appName("final_project").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 12:04:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 12:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
#import data as pandas dataframe
power_data = pd.read_csv('power_ml_data.csv')
#convert pandas dataframe to spark dataframe
power_df = spark.createDataFrame(power_data)

Using `.show()` we can see what our data columns look like while `.dtypes` lets us see what data type each column is stored as.

In [4]:
power_df.show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [5]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

## Fitting the Model
The first part of this project will be training an elastic net model with out dataset to predict values for Power Zone 3. In an elastic net model, L1 (LASSO) and L2 (Ridge) penalties are combined to improve model predictions and stability. 

Now that we've loaded our dataset and have a good idea of what our data looks like, we can set up the transformations we want to apply to our data before training our model. The first transformation we'll apply is a SQL transformation to cast the Hour variable as a double instead of an integer. Within the same transformation, we will also set the Power_Zone_3 as label. After changing the Hour variable type, we'll apply a binarizer transformation to this variable to distinguish between night and day using 6.5 as the cutoff. Next , we'll use one-hot encoding to encode the Month variable. Additionally, we will run a PCA (Principle Component Analysis) fit on a few of the columns in our dataset. The PCA entails using a VectorAssembler transformation to place the desired variables together in a column followed by using the PCA transformation.

Lastly, we will use the VectorAssembler transformation to combine our desired predictor variables in a features column. 

In [6]:
#SQL transformer to cast Hour variable as DoubleType
sqlTrans = SQLTransformer(
    statement = """
                SELECT *,
                CAST(Hour AS DOUBLE) AS hour_double,
                Power_Zone_3 as label 
                FROM __THIS__
                """)

In [7]:
#Binarize transformer to convert continuous Hour values to binary values 
binarizer = Binarizer(threshold=6.5, inputCol="hour_double", outputCol="hour_binary")

In [8]:
#One-hot encoder to transform Month values to vector values
##StringIndexer transformation to conver Month values 
indexer = StringIndexer(inputCol="Month", outputCol="month_index")
##OneHotEncoder transformation
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["month_vec"])

In [9]:
#PCA transformation 
##VectorAssembler to combine desired columns
pca_assembler = VectorAssembler(inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"], outputCol="pca_features")
##PCA transformer 
pca = PCA(k=2,inputCol="pca_features", outputCol="pca_results")

In [10]:
#VectorAssembler to put predictors in features column 
assembler_features = VectorAssembler(inputCols=["hour_binary","Power_Zone_1","Power_Zone_2","month_vec","pca_results"], outputCol="features")

Now that we have defined our transformations, we can define the other components of our model. First we'll create an object to define our linear regression model. Then we'll define our parameter grid to set test values of `regParam`, which controls the amount of regularization, and `elasticNetParam`, which defines the balance between L1 and L2 regularization. We will also set up a pipeline with the transformations defined above and our linear regression model. 

In [11]:
#define object for linear regression model 
lr = LinearRegression()
#define parameter grid 
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
#define transformation pipeline 
pipeline = Pipeline(stages = [sqlTrans, binarizer, indexer, encoder, pca_assembler, pca, assembler_features, lr])

The next step is to set up our `CrossValidator` object and enter in our pipeline, parameter grid, and RMSE regression evaluator. For our cross validation, we'll use 5 folds. Now we can fit our cross validation model.

In [12]:
#set up cross validation 
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [13]:
cvModel = crossval.fit(power_df)

26/04/28 17:10:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 17:10:07 WARN Instrumentation: [b6cdef7a] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:09 WARN Instrumentation: [b6cdef7a] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:11 WARN Instrumentation: [148b699c] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:12 WARN Instrumentation: [148b699c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

At this point, we've successfully used cross validation to train an elastic net model! Let's save this model to avoid having to rerun the training every time we reopen our notebook. We can use `.save()` to save our model in a folder, and `.load()` to reload this model when we want to.

In [14]:
cvModel.write().overwrite().save("cvModel")

In [13]:
cvModel = CrossValidatorModel.load("cvModel")

Let's see how our elastic net model performs. After cross validation, the optimal hyperparameters chosen were a `regParam` of 0.05 and an `elasticNetParam` of 0.1. 

In [33]:
#extract last training stage of the best model 
best_model = cvModel.bestModel.stages[-1]
#iterate through paramters in parameter map for the best model
print("Optimal Paramters:")
print("-" * 30)
for param, value in best_model.extractParamMap().items():
    #print regParam and elasticNetParam
    if param.name in [p.name for p in paramGrid[0].keys()]:
        print(f"{param.name}: {value}")

Optimal Paramters:
------------------------------
elasticNetParam: 0.1
regParam: 0.05


We can also take a look at the CV errors for each combination of `regParam` and `elasticNetParam`. We can see that many of the RMSE values eneded up being similar varying only in their decimal point values.

In [39]:
print("CV Errors:")
print("-" * 30)
for params, score in zip(paramGrid, cvModel.avgMetrics):
    param_str = "|".join([f"{param.name}={value}" for param, value in params.items()])
    print(f"{param_str} -- RMSE: {score:.4f}")

CV Errors:
------------------------------
regParam=0.0|elasticNetParam=0.0 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.05 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.1 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.25 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.5 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.75 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.9 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.95 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.98 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.99 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=1.0 -- RMSE: 2147.8759
regParam=0.05|elasticNetParam=0.0 -- RMSE: 2147.8758
regParam=0.05|elasticNetParam=0.05 -- RMSE: 2147.8768
regParam=0.05|elasticNetParam=0.1 -- RMSE: 2147.8751
regParam=0.05|elasticNetParam=0.25 -- RMSE: 2147.8762
regParam=0.05|elasticNetParam=0.5 -- RMSE: 2147.8753
regParam=0.05|elasticNetParam=0.75 -- RMSE: 2147.8756
regParam=0.05|elasticNetParam=0.9 -- RMSE: 2147.8755
regPar

Now we also want to calculate the resulting RMSE when we use our cvModel to predict Power_Zone_3 values of our original dataset. Doing so gives us an RMSE value of 2147.097. 

In [40]:
lr_rmse = RegressionEvaluator().evaluate(cvModel.transform(power_df))
print(f"RMSE: {lr_rmse}")

RMSE: 2147.0973169293934


Our last step with our model will be using it to add a residual column to our dataframe. First, we use our cvModel as a transformation to add a column of predicted values to our dataframe. We also create a column with residuals values showing the deviation of our predicted values from the original values. 

In [15]:
power_df_pred = cvModel.transform(power_df)
power_df_resid = power_df_pred.withColumn("residual",col("label")-col("prediction"))
power_df_resid.select("label","prediction","residual").show(8)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20878.850660788765| -637.886800788765|
|20131.08434|18660.227266544003| 1470.857073455998|
|19668.43373| 18204.75215311452|1463.6815768854794|
|18899.27711|17590.648498339124|1308.6286116608753|
|18442.40964| 16997.30198645687|1445.1076535431312|
|18130.12048| 16517.68672349429|1612.4337565057103|
|17945.06024|16093.246141053492|1851.8140989465064|
|17459.27711|15722.695360253929|1736.5817497460703|
+-----------+------------------+------------------+
only showing top 8 rows


## Streaming Data
In the previous section we trained an elastic net model on our dataset to predict values of Power_Zone_3. With this model, we can now make predictions with new data that we read in from a stream. First we define a schema for the data that will be streamed in, following the schema from our `power_df` dataframe we used in the previous section. 

In [23]:
myschema = power_df.schema

In [24]:
#read csv files from folder 'streaming_files' following schema 
stream_df = spark.readStream.schema(myschema).format("csv").option("header","true").load("streaming_files")

#transformation to rename response variable to label
rename_df = stream_df.withColumnRenamed("Power_Zone_3","label") 

In [25]:
write_rename_df = rename_df.writeStream.outputMode("append").format("console").start()

26/04/27 22:15:49 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-477c1e96-859b-4753-bfc0-2fae343c1a3a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 22:15:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [27]:
#SQL transformation to calculate residual and select only label, prediction, and residual columns 
sqlStream = SQLTransformer(
    statement = """
                SELECT prediction,
                label - prediction AS residual
                FROM __THIS__
                """)
#set up pipeline and fit pipeline to bike data
pipeline = Pipeline(stages = [cvModel, sqlStream])
df_pipeline = pipeline.fit(stream_df)

writeDF = df_pipeline.transform(stream_df).writeStream.outputMode("append").format("console").start()

26/04/27 22:16:14 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-02651633-5afa-460b-ac99-ab45fbd8758d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 22:16:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [29]:
%run produce_stream_data.py

-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
Batch: 0
-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|      13.94|    90.1|     0.069|                115.9|        106.1|  33952.3789|  18509.5723|    19392.0|    4|  13|
|      12.07|    77.0|     4.916|                0.037|        0.122| 24126.10169| 14261.39818|14837.06533|    2|   5|
|      12.35|    